# Bronze Layer Profiling — Raw Data Quality Report


**Purpose:** Analyze the three raw FinMark datasets BEFORE any cleaning. This serves as the input to the Silver Layer cleaning stage.

---

## What this notebook does

For each of the three raw datasets (`event_logs.csv`, `marketing_summary.csv`, `trend_report.csv`), we examine:

1. Row and column counts
2. Data types per column
3. Missing values (count + percentage)
4. Duplicate records
5. Inconsistent or suspect values

This is a read-only inspection — no cleaning happens here. Cleaning is done in the Silver layer.


In [1]:
import pandas as pd
from pathlib import Path

RAW = Path.cwd() / "data" / "raw"
print(f"Raw data folder: {RAW}")
print(f"Files: {[f.name for f in RAW.glob('*.csv')]}")


Raw data folder: c:\Users\sheen\OneDrive\Documents\plat_tech\ms2_submission\finmark_pipeline\data\raw
Files: ['event_logs.csv', 'marketing_summary.csv', 'trend_report.csv']


## Profiling Helper

A single function we'll call on each dataset to keep the output consistent.


In [2]:
def profile_dataset(filename: str) -> dict:
    """Load a CSV and produce a structured profiling report."""
    print(f"\n{'='*70}")
    print(f"PROFILING: {filename}")
    print('='*70)

    df = pd.read_csv(RAW / filename)

    # Shape
    print(f"\nShape: {df.shape[0]} rows x {df.shape[1]} columns")

    # Column listing
    meaningful_cols = [c for c in df.columns if not c.startswith("col_")]
    junk_cols = [c for c in df.columns if c.startswith("col_")]
    print(f"\nMeaningful columns ({len(meaningful_cols)}): {meaningful_cols}")
    print(f"Junk/unnamed columns ({len(junk_cols)}): col_? through col_{len(junk_cols)+5 if junk_cols else 0}")

    # Data types (meaningful columns only)
    print(f"\nData types of meaningful columns:")
    print(df[meaningful_cols].dtypes.to_string())

    # Missing values
    missing = df[meaningful_cols].isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
    missing_df = missing_df[missing_df["missing_count"] > 0]
    if len(missing_df) > 0:
        print(f"\nMissing values in meaningful columns:")
        print(missing_df.to_string())
    else:
        print("\nNo missing values in meaningful columns.")

    # Duplicates
    dup_count = df.duplicated().sum()
    print(f"\nFully duplicated rows: {dup_count}")

    # Sample
    print(f"\nFirst 3 rows (meaningful columns only):")
    print(df[meaningful_cols].head(3).to_string())

    return {
        "filename": filename,
        "rows": df.shape[0],
        "cols": df.shape[1],
        "meaningful_cols": len(meaningful_cols),
        "junk_cols": len(junk_cols),
        "missing_values": int(missing.sum()),
        "duplicates": int(dup_count),
    }


## Dataset 1: `event_logs.csv`

User interaction events on the FinMark platform.


In [3]:
profile_events = profile_dataset("event_logs.csv")



PROFILING: event_logs.csv

Shape: 2000 rows x 50 columns

Meaningful columns (5): ['user_id', 'event_type', 'event_time', 'product_id', 'amount']
Junk/unnamed columns (45): col_? through col_50

Data types of meaningful columns:
user_id        object
event_type     object
event_time     object
product_id     object
amount        float64

Missing values in meaningful columns:
        missing_count  missing_pct
amount           1016         50.8

Fully duplicated rows: 0

First 3 rows (meaningful columns only):
  user_id      event_type        event_time product_id   amount
0   U0099        checkout  2023-06-03 04:13       P010      NaN
1   U0240    wishlist_add  2023-06-03 05:08       P020  2900.63
2   U0374  profile_update  2023-06-05 06:22       P028      NaN


### Suspect values in event_logs

Beyond the basic profile, we check for two specific concerns:

1. **Null amounts on checkout events** — a transaction event should have an amount; if it doesn't, the data is incomplete.
2. **Event type distribution** — real user behavior usually shows steep drop-off between event types; if counts are too uniform, it suggests sampling or synthetic data.


In [4]:
df_events = pd.read_csv(RAW / "event_logs.csv")

# 1. Checkout events with null amounts
checkout_null = df_events[(df_events["event_type"] == "checkout") & (df_events["amount"].isnull())]
total_checkout = (df_events["event_type"] == "checkout").sum()
print(f"Checkout events: {total_checkout}")
print(f"Checkout events with NULL amount: {len(checkout_null)}")
print(f"Percentage of checkouts missing amount: {len(checkout_null)/total_checkout*100:.1f}%")

# 2. Event type distribution
print("\nEvent type counts:")
print(df_events["event_type"].value_counts().to_string())
print("\nNote: Counts are unusually uniform — real user behavior shows steep drop-off (e.g. far more page_views than checkouts).")


Checkout events: 260
Checkout events with NULL amount: 142
Percentage of checkouts missing amount: 54.6%

Event type counts:
event_type
login             276
checkout          260
wishlist_add      260
profile_update    255
add_to_cart       252
search            251
page_view         233
logout            213

Note: Counts are unusually uniform — real user behavior shows steep drop-off (e.g. far more page_views than checkouts).


## Dataset 2: `marketing_summary.csv`

Daily marketing performance summary.


In [5]:
profile_marketing = profile_dataset("marketing_summary.csv")



PROFILING: marketing_summary.csv

Shape: 100 rows x 50 columns

Meaningful columns (5): ['date', 'users_active', 'total_sales', 'new_customers', 'report_generated']
Junk/unnamed columns (45): col_? through col_50

Data types of meaningful columns:
date                 object
users_active          int64
total_sales         float64
new_customers         int64
report_generated     object

No missing values in meaningful columns.

Fully duplicated rows: 0

First 3 rows (meaningful columns only):
         date  users_active  total_sales  new_customers  report_generated
0  2023-06-01           179     81287.31              9  2023-06-01 16:00
1  2023-06-02            67     74771.99              5  2023-06-02 16:00
2  2023-06-03           369     84809.74             11  2023-06-03 16:00


### Batch generation pattern in marketing_summary

We check the `report_generated` column to confirm whether the report is generated in real-time or as a daily batch. From our Milestone 1 analysis, we expect every value to be at hour 16.


In [6]:
df_marketing = pd.read_csv(RAW / "marketing_summary.csv")
df_marketing["report_generated"] = pd.to_datetime(df_marketing["report_generated"], errors="coerce")

hours = df_marketing["report_generated"].dt.hour.dropna().unique()
print(f"Unique hours in report_generated: {sorted(hours)}")
print(f"Number of unique hours: {len(hours)}")
print("\nIf all values are at the same hour, this confirms batch generation (not real-time).")
print("\nSample of report_generated timestamps:")
print(df_marketing["report_generated"].head(5).to_string())


Unique hours in report_generated: [np.int32(16)]
Number of unique hours: 1

If all values are at the same hour, this confirms batch generation (not real-time).

Sample of report_generated timestamps:
0   2023-06-01 16:00:00
1   2023-06-02 16:00:00
2   2023-06-03 16:00:00
3   2023-06-04 16:00:00
4   2023-06-05 16:00:00


## Dataset 3: `trend_report.csv`

Weekly trend report.


In [7]:
profile_trends = profile_dataset("trend_report.csv")



PROFILING: trend_report.csv

Shape: 20 rows x 50 columns

Meaningful columns (3): ['week', 'avg_users', 'sales_growth_rate']
Junk/unnamed columns (47): col_? through col_52

Data types of meaningful columns:
week                  object
avg_users              int64
sales_growth_rate    float64

No missing values in meaningful columns.

Fully duplicated rows: 0

First 3 rows (meaningful columns only):
       week  avg_users  sales_growth_rate
0  2023-W21        328             -0.003
1  2023-W22        280              0.088
2  2023-W23        130              0.073


### Junk column content in trend_report

The trend_report has the worst junk-column problem (47 unnamed columns). Let's peek at what's actually in them to confirm they have no usable structure.


In [8]:
df_trends = pd.read_csv(RAW / "trend_report.csv")
junk_cols = [c for c in df_trends.columns if c.startswith("col_")]
print(f"Number of junk columns: {len(junk_cols)}")
print(f"First 10 junk column names: {junk_cols[:10]}")
print("\nSample values from first 5 junk columns (rows 0-4):")
print(df_trends[junk_cols[:5]].head().to_string())
print("\nUnique value types per junk column (first 5):")
for col in junk_cols[:5]:
    types = df_trends[col].dropna().apply(type).unique()
    print(f"  {col}: types = {[t.__name__ for t in types]}")
print("\nVerdict: Junk columns have mixed types (str + float) with no documentation. Cannot be used safely. Will be dropped in Silver.")


Number of junk columns: 47
First 10 junk column names: ['col_4', 'col_5', 'col_6', 'col_7', 'col_8', 'col_9', 'col_10', 'col_11', 'col_12', 'col_13']

Sample values from first 5 junk columns (rows 0-4):
     col_4   col_5    col_6    col_7   col_8
0      NaN     NaN   Stable      NaN  2751.0
1      NaN     NaN  Falling    16.14  2201.0
2  1231.77  1381.0      NaN  1656.94     NaN
3   184.26     NaN      NaN   902.57   982.0
4      NaN   243.0   Rising   514.97   329.0

Unique value types per junk column (first 5):
  col_4: types = ['float']
  col_5: types = ['float']
  col_6: types = ['str']
  col_7: types = ['float']
  col_8: types = ['float']

Verdict: Junk columns have mixed types (str + float) with no documentation. Cannot be used safely. Will be dropped in Silver.


## Summary: Raw Data Quality Report

This is the consolidated output for the Silver Layer cleaning stage.


In [10]:
import pandas as pd

summary = pd.DataFrame([profile_events, profile_marketing, profile_trends])
summary["pct_missing_overall"] = (summary["missing_values"] / (summary["rows"] * summary["meaningful_cols"]) * 100).round(2)
print("Raw data quality summary across all three datasets:")
print(summary.to_string(index=False))


Raw data quality summary across all three datasets:
             filename  rows  cols  meaningful_cols  junk_cols  missing_values  duplicates  pct_missing_overall
       event_logs.csv  2000    50                5         45            1016           0                10.16
marketing_summary.csv   100    50                5         45               0           0                 0.00
     trend_report.csv    20    50                3         47               0           0                 0.00


## Findings handed off to (Silver Layer)

These are the issues Silver Layer cleaning needs to address:

### `event_logs.csv` (2,000 rows × 50 cols)
- **45 junk columns** (`col_6` to `col_50`) with no documentation → drop
- **142 checkout events have NULL amounts** (54.6% of checkouts) → keep rows but flag
- **Event counts are uniformly distributed** (login 276 → logout 213) → cannot be fixed in cleaning, escalate as data source issue
- `event_time` is loaded as string → coerce to datetime

### `marketing_summary.csv` (100 rows × 50 cols)
- **45 junk columns** → drop
- **`report_generated` is always at hour 16** → confirms batch-only architecture from Milestone 1
- `date` and `report_generated` are loaded as strings → coerce to datetime
- No nulls in meaningful columns ✅

### `trend_report.csv` (20 rows × 50 cols)
- **47 junk columns** (`col_4` to `col_50`) with mixed string/numeric content → drop
- Only 3 meaningful columns survive: `week`, `avg_users`, `sales_growth_rate`
- No nulls in meaningful columns ✅
- Very short historical window (only 20 weeks) → noted as a limitation, not a cleaning issue

---

## What's NOT being fixed here (out of scope for Bronze profiling)

- Cleaning (Silver stage)
- Aggregations and KPIs (Gold stage)
- Data dictionary (separate `DATA_DICTIONARY.md` file)
- Predictive modeling (out of scope for Milestone 2)
